# Template Inversion Evaluation

This notebook evaluates the robustness of the proposed anonymization
algorithms against template inversion attacks.

A trained SciFive-based embedding inversion model is used to reconstruct
clinical text from anonymized embeddings.

The reconstruction quality is evaluated using:

- BLEU
- BERTScore-F1

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

from src.inversion import (
    load_or_train_inversion_model,
    evaluate_embeddings,
)

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"

EMBEDDING_PATH = DATA_DIR / "case_embeddings.parquet"
TEXT_PATH = DATA_DIR / "case_texts.parquet"

ANON_DIR = DATA_DIR / "anonymized_outputs"

RESULTS_DIR = DATA_DIR / "results" / "template_inversion"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = PROJECT_ROOT / "saved_inversion_model"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
embeddings = pd.read_parquet(EMBEDDING_PATH).to_numpy(dtype=np.float32)
print(embeddings.shape)

texts_df = pd.read_parquet(TEXT_PATH)[:100]
print(texts_df.shape)

In [ ]:
def truncate_to_core(
    text,
    max_sentences=5,
):
    sentences = [
        s.strip()
        for s in text.split(".")
        if len(s.strip()) > 20
    ]

    return ". ".join(
        sentences[:max_sentences]
    ) + "."


texts_df["case_text"] = (
    texts_df["case_text"]
    .fillna("")
)

texts_df["truncated_text"] = (
    texts_df["case_text"]
    .apply(truncate_to_core)
)

mapping_path = DATA_DIR / "text_mapping.csv"

texts_df[
    [
        "case_id",
        "case_text",
        "truncated_text",
    ]
].to_csv(
    mapping_path,
    index=False,
)

texts = texts_df[
    "truncated_text"
].tolist()

print(
    "Total texts:",
    len(texts),
)

print("\nExample:\n")
print(texts[0])

In [ ]:
tokenizer, model, adapter, device = (
    load_or_train_inversion_model(
        embeddings=embeddings,
        texts=texts,
        model_dir=MODEL_DIR,
        epochs=50,
        batch_size=16,
        learning_rate=3e-4,
        latent_len=48,
        device=device,
    )
)

In [ ]:
summary = []

summary.append(
    evaluate_embeddings(
        embeddings=embeddings,
        reference_texts=texts,
        tokenizer=tokenizer,
        model=model,
        adapter=adapter,
        device=device,
        label="Baseline",
        output_dir=RESULTS_DIR / "baseline",
    )
)

In [ ]:
CLUSTER_K = [5, 10, 25, 50, 100]

for k in CLUSTER_K:

    print(f"\nMDAV-C | k={k}")

    anon_embeddings = (
        pd.read_parquet(
            ANON_DIR /
            f"mdav_c_k{k}.parquet"
        )
        .to_numpy(dtype=np.float32)
    )

    summary.append(
        evaluate_embeddings(
            embeddings=anon_embeddings,
            reference_texts=texts,
            tokenizer=tokenizer,
            model=model,
            adapter=adapter,
            device=device,
            label=f"MDAV-C-k{k}",
            output_dir=(
                RESULTS_DIR /
                "mdav_c" /
                f"k_{k}"
            ),
        )
    )

In [ ]:
for k in CLUSTER_K:

    print(f"\nRMDAV-M | k={k}")

    anon_embeddings = (
        pd.read_parquet(
            ANON_DIR /
            f"rmdav_m_k{k}.parquet"
        )
        .to_numpy(dtype=np.float32)
    )

    summary.append(
        evaluate_embeddings(
            embeddings=anon_embeddings,
            reference_texts=texts,
            tokenizer=tokenizer,
            model=model,
            adapter=adapter,
            device=device,
            label=f"RMDAV-M-k{k}",
            output_dir=(
                RESULTS_DIR /
                "rmdav_m" /
                f"k_{k}"
            ),
        )
    )

In [ ]:
summary_df = pd.DataFrame(summary)

summary_df.to_csv(
    RESULTS_DIR /
    "summary.csv",
    index=False,
)

summary_df